In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.nn.utils.rnn import pad_sequence


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using Device: {device}")


Using Device: cuda


In [ ]:
base_path = "/content/drive/MyDrive/features"

real_train = sorted(os.listdir(os.path.join(base_path, "train/real")))
fake_train = sorted(os.listdir(os.path.join(base_path, "train/fake")))

real_val = sorted(os.listdir(os.path.join(base_path, "validate/real")))
fake_val = sorted(os.listdir(os.path.join(base_path, "validate/fake")))

real_test = sorted(os.listdir(os.path.join(base_path, "test/real")))
fake_test = sorted(os.listdir(os.path.join(base_path, "test/fake")))

print(f"Training: {len(real_train)} real, {len(fake_train)} fake")
print(f"Validation: {len(real_val)} real, {len(fake_val)} fake")
print(f"Testing: {len(real_test)} real, {len(fake_test)} fake")


Training: 216 real, 1839 fake
Validation: 72 real, 613 fake
Testing: 75 real, 616 fake


In [ ]:
class DeepFakeDataset(Dataset):
    def __init__(self, video_list, label, base_path):
        self.video_list = video_list
        self.label = label
        self.base_path = base_path

    def __len__(self):
        return len(self.video_list)

    def __getitem__(self, idx):
        video_name = self.video_list[idx]
        feature_path = os.path.join(self.base_path, video_name)

        if os.path.exists(feature_path) and os.path.getsize(feature_path) > 0:
            features = np.load(feature_path)
        else:
            print(f"❌ Error: Missing or empty file: {feature_path}")
            features = np.zeros((1, 1792), dtype=np.float32)

        if features.shape[-1] != 1792:
            print(f"⚠️ Warning: {video_name} shape mismatch {features.shape}")
            features = features.reshape(-1, 1792)

        return torch.tensor(features, dtype=torch.float32), torch.tensor(self.label, dtype=torch.long)


In [ ]:
train_dataset = ConcatDataset([
    DeepFakeDataset(real_train, 0, os.path.join(base_path, "train/real")),
    DeepFakeDataset(fake_train, 1, os.path.join(base_path, "train/fake"))
])

val_dataset = ConcatDataset([
    DeepFakeDataset(real_val, 0, os.path.join(base_path, "validate/real")),
    DeepFakeDataset(fake_val, 1, os.path.join(base_path, "validate/fake"))
])

test_dataset = ConcatDataset([
    DeepFakeDataset(real_test, 0, os.path.join(base_path, "test/real")),
    DeepFakeDataset(fake_test, 1, os.path.join(base_path, "test/fake"))
])


In [ ]:
def collate_fn(batch):
    features, labels = zip(*batch)
    padded_features = pad_sequence(features, batch_first=True)  # pad to max length
    labels = torch.tensor(labels)
    return padded_features, labels


In [ ]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)


In [ ]:
class Attention(nn.Module):
    def __init__(self, hidden_dim):
        super(Attention, self).__init__()
        self.attn = nn.Linear(hidden_dim, 1)
        self.softmax = nn.Softmax(dim=1)

    def forward(self, lstm_out):
        attn_weights = self.softmax(self.attn(lstm_out))
        context = torch.sum(attn_weights * lstm_out, dim=1)
        return context

class DeepFakeModel(nn.Module):
    def __init__(self, hidden_size=512, num_layers=2):
        super(DeepFakeModel, self).__init__()
        self.lstm = nn.LSTM(input_size=1792, hidden_size=hidden_size, num_layers=num_layers, batch_first=True)
        self.attention = Attention(hidden_size)
        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        attn_out = self.attention(lstm_out)
        return self.fc(attn_out)


In [ ]:
param_grid = {
    'learning_rate': [0.001, 0.0001, 0.01],
    'num_layers': [1, 2, 3],
    'hidden_size': [256, 512, 1024]
}

best_model = None
best_val_acc = 0.0
best_params = {}

for lr in param_grid['learning_rate']:
    for num_layers in param_grid['num_layers']:
        for hidden_size in param_grid['hidden_size']:
            print(f"\n🔧 Training with LR: {lr}, Layers: {num_layers}, Hidden: {hidden_size}")

            model = DeepFakeModel(hidden_size=hidden_size, num_layers=num_layers).to(device)
            optimizer = optim.Adam(model.parameters(), lr=lr)
            criterion = nn.CrossEntropyLoss()
            scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

            for epoch in range(50):
                model.train()
                total_loss, correct, total = 0, 0, 0

                for features, labels in train_loader:
                    features, labels = features.to(device), labels.to(device)
                    optimizer.zero_grad()
                    outputs = model(features)
                    loss = criterion(outputs, labels)

                    try:
                        loss.backward()
                        optimizer.step()
                    except RuntimeError:
                        print("⚠️ Skipped batch due to CUDA OOM")
                        continue

                    total_loss += loss.item()
                    correct += (outputs.argmax(dim=1) == labels).sum().item()
                    total += labels.size(0)

                train_acc = correct / total

                # Validation
                model.eval()
                val_correct, val_total = 0, 0
                with torch.no_grad():
                    for features, labels in val_loader:
                        features, labels = features.to(device), labels.to(device)
                        outputs = model(features)
                        val_correct += (outputs.argmax(dim=1) == labels).sum().item()
                        val_total += labels.size(0)

                val_acc = val_correct / val_total
                print(f"Epoch {epoch+1}/50 | Loss: {total_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

                if val_acc > best_val_acc:
                    best_val_acc = val_acc
                    best_params = {'lr': lr, 'batch_size': 16, 'num_layers': num_layers, 'hidden_size': hidden_size}
                    best_model = model.state_dict()
                    print("✅ Best model updated!")

                scheduler.step()

print("\n🎉 Grid Search Completed!")
print(f"🏆 Best Hyperparameters: {best_params}")



🔧 Training with LR: 0.001, Layers: 1, Hidden: 256
Epoch 1/50 | Loss: 45.2886 | Train Acc: 0.8944 | Val Acc: 0.8949
✅ Best model updated!
Epoch 2/50 | Loss: 37.6265 | Train Acc: 0.9022 | Val Acc: 0.8978
✅ Best model updated!
Epoch 3/50 | Loss: 30.8111 | Train Acc: 0.9144 | Val Acc: 0.9197
✅ Best model updated!
Epoch 4/50 | Loss: 25.4951 | Train Acc: 0.9251 | Val Acc: 0.9226
✅ Best model updated!
Epoch 5/50 | Loss: 22.8302 | Train Acc: 0.9285 | Val Acc: 0.9197
Epoch 6/50 | Loss: 18.5483 | Train Acc: 0.9431 | Val Acc: 0.9416
✅ Best model updated!
Epoch 7/50 | Loss: 18.3202 | Train Acc: 0.9445 | Val Acc: 0.9416
Epoch 8/50 | Loss: 14.9981 | Train Acc: 0.9596 | Val Acc: 0.9314
Epoch 9/50 | Loss: 14.6672 | Train Acc: 0.9562 | Val Acc: 0.9109
Epoch 10/50 | Loss: 14.2703 | Train Acc: 0.9596 | Val Acc: 0.9416
Epoch 11/50 | Loss: 10.8908 | Train Acc: 0.9679 | Val Acc: 0.9562
✅ Best model updated!
Epoch 12/50 | Loss: 11.7858 | Train Acc: 0.9684 | Val Acc: 0.9533
Epoch 13/50 | Loss: 8.6450 | Train